In [1]:
# ---- version 2.0 ------
# Read linkedin pdf
# Read summary document
# compile system prompt and concatenate the linkedin & summary details with it as context
# invoke openai api and send the user messages, the usual way
# If user provides email id then tool-call or unknown-question tool call
# in tools push notification to phone
# capture the overall response
# use gradio as UI

In [2]:
import os
import json
import requests
import gradio as gr

from dotenv import load_dotenv
from pypdf import PdfReader
from openai import OpenAI
from pydantic import BaseModel

In [3]:
load_dotenv(override=True)
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
pushover_url = "https://api.pushover.net/1/messages.json"
# payload = {"user": pushover_user, "token": pushover_token, "message": "Hi!"}

In [5]:
# requests.post(pushover_url, data=payload)

In [6]:
def push(msg):
    print(f"{msg}")
    payload = {"user": pushover_user, "token": pushover_token, "message": msg}
    requests.post(pushover_url, data=payload)

In [7]:
def record_user_details(email, name="not provided", notes="not provided"):
    push(f"Recording interest from {name} with email: {email} and notes: {notes}")
    return {"recorded": "ok"}

In [8]:
def record_unknown_question(question):
    push(f"Recording {question} that I couldn't answer")
    return {"recorded": "ok"}

In [9]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(tool_call)
        print()
        print(f"Tool called: {tool_name}", flush=True)
        
        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [10]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "email address of this user"
            },
            "name": {
                "type": "string",
                "description": "name of this user if provided"
            },
            "notes": {
                "type": "string",
                "description": "any additional information that is worth ading for better context"
            }
        }
    },
    "required": ["email"],
    "additionalProperties": False
}

In [11]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": " Use this tool to record any unkwown question that couldn't be answered as you don't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            }
        }
    },
    "required": ["question"],
    "additionalProperties": False
}

In [12]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'email address of this user'},
     'name': {'type': 'string',
      'description': 'name of this user if provided'},
     'notes': {'type': 'string',
      'description': 'any additional information that is worth ading for better context'}}},
   'required': ['email'],
   'additionalProperties': False}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': " Use this tool to record any unkwown question that couldn't be answered as you don't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}}},
   'required': ['question'],
   'additionalProperties': Fals

In [13]:
reader = PdfReader("linkedin.pdf")

linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [14]:
with open("summary.txt", 'r', encoding='utf-8') as f:
    summary = f.read()

In [15]:
name = "Debmalya Mondal"

In [16]:
system_prompt = f"You are acting as {name}. You introduce yourself as {name}'s digital avatar. You are answering questions on behalf \
of {name} about {name}'s career, background, skills, and experience. Your responsibility to provide the website visitors with \
valid information and represent {name} as honest and truthful as possible. Be professional and engaging in your responses as you may \
be speaking to a potential employer or customer. If you don't know the answer say so. If the user does not ask questions about {name}'s \
career or skills, then humbly ask the user to stick to the topic and deny answering. Use your record_unknown_question tool to record \
the question that you couldn't answer. If the user is engaging in discussion, try to steer them in towards getting in touch via email; \
and use record_user_details tool to record it. Do not ask if name is not provided. After sharing email address, if the user asks to get in touch, \
tell the user that you will definitely get in touch as soon as possible."

system_prompt += f"\n\n##Summary:\n{summary}\n\n"
system_prompt += f"\n\n##LinkedIn: \n{linkedin}\n\n"
system_prompt += "\n\nUsing this context, chat with the user"

In [17]:
openai = OpenAI(api_key=openai_api_key)

In [18]:
def chat(message, history=[]):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            print(results)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [19]:
# chat("can you share your resume at debmalya@gmail.com?")

In [20]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ChatCompletionMessageFunctionToolCall(id='call_ilkVaWihhLf569xUIMgcm0ac', function=Function(arguments='{"question":"who is mandella"}', name='record_unknown_question'), type='function')

Tool called: record_unknown_question
Recording who is mandella that I couldn't answer
[{'role': 'tool', 'content': '{"recorded": "ok"}', 'tool_call_id': 'call_ilkVaWihhLf569xUIMgcm0ac'}]
ChatCompletionMessageFunctionToolCall(id='call_REDwARbZBHf4OCyYfHvt4bPP', function=Function(arguments='{"email":"you@google.com"}', name='record_user_details'), type='function')

Tool called: record_user_details
Recording interest from not provided with email: you@google.com and notes: not provided
[{'role': 'tool', 'content': '{"recorded": "ok"}', 'tool_call_id': 'call_REDwARbZBHf4OCyYfHvt4bPP'}]
